# 11. MCP (Model Context Protocol)

## 학습 목표

MCP(Model Context Protocol)를 통해 외부 도구와 컨텍스트를 에이전트에 연결하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- MCP의 개념과 아키텍처(서버/클라이언트/호스트)를 이해한다
- `langchain-mcp-adapters` 패키지로 MCP 서버에 연결한다
- `ChatOpenAI.bind_tools(mcp_tools)`로 에이전트와 MCP 도구를 통합한다
- Stdio와 SSE 전송 방식의 차이를 안다
- 다중 MCP 서버를 연결하는 방법을 익힌다

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — 


In [3]:
import logging
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

## 11.2 MCP 개념

**MCP(Model Context Protocol)**는 외부 도구와 컨텍스트를 **표준화된 방식**으로 LLM에 제공하기 위한 오픈 프로토콜입니다.

### 아키텍처 구성 요소

| 구성 요소 | 역할 | 예시 |
|----------|------|------|
| **MCP 서버** | 도구, 리소스, 프롬프트를 노출 | 파일 시스템 서버, DB 서버, API 래퍼 |
| **MCP 클라이언트** | 서버에 연결하여 도구를 가져옴 | `MultiServerMCPClient` |
| **호스트** | 클라이언트를 관리하고 LLM과 연결 | LangChain 에이전트, IDE |

### 핵심 리소스 타입

- **Tools**: 에이전트가 호출할 수 있는 실행 가능한 함수
- **Resources**: 파일, DB 레코드 등의 데이터 (LangChain Blob 객체로 변환)
- **Prompts**: 재사용 가능한 프롬프트 템플릿

### 왜 MCP인가?

MCP 이전에는 각 도구마다 개별적으로 연결 코드를 작성해야 했습니다. MCP는 이를 **하나의 표준 프로토콜**로 통합하여:
- 도구 제공자는 한 번만 MCP 서버를 구현하면 됩니다
- LLM 호스트는 MCP 클라이언트 하나로 모든 도구에 접근할 수 있습니다
- 생태계 전체에서 도구를 재사용할 수 있습니다

## 다중 MCP 서버 연결

`MultiServerMCPClient`는 이름 그대로 여러 MCP 서버를 동시에 관리할 수 있습니다.

In [4]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent

client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",  # Local subprocess communication
            "command": "python",
            # Absolute path to your math_server.py file
            "args": ["./math_server.py"],
        },
        "weather": {
            "transport": "http",  # HTTP-based remote server
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()
agent = create_agent(
    "openai:gpt-5.4-mini",
    tools  
)
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]},
    config=lf_config
)
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]},
    config=lf_config
)
print(math_response['messages'][0].content)
print(weather_response['messages'][0].content)

what's (3 + 5) x 12?
what is the weather in nyc?
